# Module 2 — Persistent vector search with sqlitesearch (lesson 07)

minsearch has three limits for vector search: it re-embeds everything on startup (~1 min here), keeps everything in
memory, and scores every document (brute force). sqlitesearch stores vectors in a SQLite file and uses
**approximate nearest neighbour (ANN)** search: narrow to a region of likely matches, then rerank only those exactly.

```
NN  (exact):  compare query with ALL documents            -> top 5
ANN (approx): pick candidate region -> compare inside it -> top 5
```

## Ingestion process: embed once, write to disk

In [1]:
import os
import time

import numpy as np
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex
from tqdm.auto import tqdm

from ingest import load_faq_data

model = SentenceTransformer("all-MiniLM-L6-v2")
documents = load_faq_data()
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

X = np.array([v for i in tqdm(range(0, len(texts), 50)) for v in model.encode(texts[i:i + 50])])
X.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

(1489, 384)

In [2]:
DB_PATH = "faq_vectors.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)   # notebook re-runs start from a clean index

vs_index = VectorSearchIndex(keyword_fields=["course"], mode="ivf", db_path=DB_PATH)
vs_index.fit(X, documents)
print("indexed:", vs_index.count())
vs_index.close()

indexed: 1489


## Query process: reopen without re-embedding

A fresh index object pointed at the same file, with no `fit` call. Only the *query* is embedded at request time,
which is why the query side starts instantly while ingestion pays the one-minute cost once.

In [3]:
vs_index = VectorSearchIndex(keyword_fields=["course"], mode="ivf", db_path=DB_PATH)

query = "I just discovered the course. Can I still join it?"
results = vs_index.search(model.encode(query), filter_dict={"course": "llm-zoomcamp"}, num_results=5)
[r["question"] for r in results]

['I just discovered the course. Can I still join?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'Can I run the course locally instead of Codespaces?',
 'Does the course certificate show the number of course hours?',
 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?']

## RAG over the persistent index

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase

load_dotenv()
groq_client = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")


class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        return self.index.search(
            self.embedder.encode(query),
            num_results=num_results,
            filter_dict={"course": self.course},
        )


assistant = RAGVector(embedder=model, index=vs_index, llm_client=groq_client)
print(assistant.rag("the program has already begun, can I still sign up?"))
vs_index.close()

Yes—you can still join even though the program has already started. If you’d like to earn a certificate, make sure you submit your capstone project before the submission window closes. Otherwise, you can begin learning and completing the homework right away.


## Extra — how approximate is "approximate"?

The lesson picks `mode="ivf"` and says `lsh` is fine for small data. The library's current default is `hnsw`.
This cell measures each mode instead of trusting either claim.

**Recall@5** = share of the exact top-5 (numpy brute force) that the ANN index also returns, averaged over 200
queries. The queries are real FAQ questions, embedded on their own.

In [5]:
rng = np.random.default_rng(42)
sample = rng.choice(len(documents), size=200, replace=False)
Q = model.encode([documents[i]["question"] for i in sample])

exact_top5 = [set(np.argsort(-X.dot(q))[:5]) for q in Q]
id_to_pos = {doc["id"]: pos for pos, doc in enumerate(documents)}

rows = []
for mode in ["lsh", "ivf", "hnsw"]:
    path = f"bench_{mode}.db"
    if os.path.exists(path):
        os.remove(path)
    index = VectorSearchIndex(keyword_fields=["course"], mode=mode, db_path=path, seed=42)

    t0 = time.perf_counter()
    index.fit(X, documents)
    build_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    found = [index.search(q, num_results=5) for q in Q]
    query_ms = (time.perf_counter() - t0) / len(Q) * 1000

    recall = np.mean([
        len({id_to_pos[r["id"]] for r in res} & exact) / 5
        for res, exact in zip(found, exact_top5)
    ])
    rows.append((mode, recall, build_s, query_ms))
    index.close()
    os.remove(path)

print(f"{'mode':<6}{'recall@5':>10}{'build s':>10}{'query ms':>10}")
for mode, recall, build_s, query_ms in rows:
    print(f"{mode:<6}{recall:>10.3f}{build_s:>10.1f}{query_ms:>10.1f}")

mode    recall@5   build s  query ms
lsh        0.519       0.2       0.9
ivf        0.876       0.1       0.2
hnsw       0.998       2.1       0.5


**Observed** (1,489 docs, 200 queries, default parameters):

| mode | recall@5 | what it does |
|------|----------|--------------|
| `lsh`  | ~0.52 | random hyperplanes hash vectors into buckets; neighbours can fall on different sides of a hyperplane |
| `ivf`  | ~0.88 | k-means clusters; only the `n_probe_clusters=4` nearest clusters are searched |
| `hnsw` | ~1.00 | proximity graph walked greedily; costs the most to build |

1. "Approximate" can mean **half the true neighbours are missing**: LSH with defaults lost ~48% of them.
   At this size brute force takes well under a millisecond, so ANN buys nothing and costs recall.
2. ANN quality depends on **tuning parameters** (`hash_size`, `n_probe_clusters`, `ef_search`), not only on the algorithm name.
   Always measure recall against exact search on your own data before trusting an index.

The same failure mode, an IVF index searching too few clusters, shows up again with pgvector in the next notebook.